# 11 | M6 — Swin UNETR: training, model selection, final test

**BraTS2020 · hồ sơ nghiên cứu M1–M6 · bản đọc/đối chiếu (clean review edition)**

> **Quan trọng:** Notebook này là bản trình bày có cấu trúc để đọc và review, **không phải Notebook tái chạy toàn bộ**. Mã thực nghiệm nguyên gốc đã được giữ trong `original_notebooks/11-original.ipynb`. Các phần quan trọng được trích nguyên văn vào `scripts/historical_cells/11/`. Không dùng **Run All** trên notebook gốc: một số cell có thể huấn luyện, tạo Dataset hoặc chạy test thật.

- Thí nghiệm: **M6** — Swin UNETR.
- Nguồn: `11-swin-unetr-training.ipynb`; SHA256 của Notebook gốc: `c848ae0a459ba50fc537ee63bff5688a131d6901f9c0ab22bee82a638a35e5f3`.
- Git pin của pipeline dùng chung: `79a95a08f11694278114a89348cf8303a1e39d67` (branch `experiment/unet3d-optimization`).


## 1. Câu hỏi nghiên cứu & thay đổi được kiểm soát

Thay đổi kiến trúc sang MONAI Swin UNETR, khởi tạo từ đầu (không dùng checkpoint M1).

**Tập dữ liệu cố định:** BraTS2020, 257 train / 55 internal validation / 56 reserved test, split seed 42. Đầu vào 4 MRI modalities (T1, T1ce, T2, FLAIR), đầu ra 4 logits (background và ba nhãn u đã remap). Train patch `96³`; full-volume inference ROI `96³`, overlap 0.25, Gaussian blending, CPU stitching.

**Riêng M6:** 20 epoch khởi tạo ngẫu nhiên, không tiếp tục checkpoint M1.


## 2. Thiết kế kỹ thuật

- Huấn luyện 20 epoch từ đầu, 15,705,646 tham số, batch 1 patch 96³, feature_size=24, use_checkpoint=True.
- M6 được chọn theo 55 ca validation, checkpoint epoch 20 đã khóa trước khi đánh giá test.
- Final test là 56 ca nội bộ, không phải test set chính thức của BraTS challenge; repo có dữ liệu test lịch sử.

**Mã tham chiếu chính:**
- `src/models/swin_unetr.py (new factory)`
- `src/evaluation/validation.py`
- `src/training/engine.py`


## 3. Điểm khởi tạo / API cốt lõi (minh họa, không tự chạy)

Đoạn dưới giúp đọc kiến trúc và khác biệt chính, không tự khởi động training hoặc truy cập test:

```python
from src.models.swin_unetr import build_swin_unetr
# Factory module được tái cấu trúc sau thực nghiệm từ cấu hình M6 đã khóa.
# Không tự động train, test hoặc thay đổi checkpoint.
```


## 4. Kết quả internal validation

**55 bệnh nhân · epoch 20 · macro-average Dice theo từng bệnh nhân.**

| WT | TC | ET | Mean Dice |
|---:|---:|---:|---:|
|0.870206185 | 0.760337431 | 0.697428657 | 0.775990758 |

- M6 Mean Dice validation = 0.775990758 (55 bệnh nhân).
- Final test Mean Dice = 0.801407077 (56 bệnh nhân).
- Bootstrap 95% CI Mean Dice test = [0.762349958, 0.836852589].
- M6 − M5 validation mean delta = +0.009303484; Wilcoxon Holm p=0.0216546; paired bootstrap CI chứa 0.

**Cách đọc:** Dice tốt hơn không đồng nghĩa có ý nghĩa thống kê; so sánh ghép cặp và Holm được lưu trong báo cáo phân tích M5.7 / M6.11.6.


## 5. Training và sao lưu đã được thực hiện như thế nào?

- Các cell huấn luyện gốc được triển khai trên Kaggle và lưu checkpoint/optimizer/AMP (tùy thí nghiệm), không lưu `.pt` trên GitHub.
- Sau khi chạy, các bản backup được xác minh bằng SHA256; kết quả tổng hợp từ những lần chạy đã hoàn tất được bảo quản trên Kaggle.
- Đừng coi đoạn code lưu trong `scripts/historical_cells/` là API thư viện chạy độc lập: chúng phụ thuộc biến trạng thái và mount của Kaggle, cần đưa vào module và kiểm thử trước khi tái sử dụng.
- Các lỗi và bước phục hồi lịch sử được giữ trong notebook gốc nhưng không nằm trong luồng đọc chính.


## 6. Chỉ mục mã nguồn gốc để audit

Những cell sau đã được trích **nguyên văn** vào thư mục `scripts/historical_cells/11/`. Cell index tính từ **0** trong file Notebook gốc.

| Nội dung | Cell index | File gốc đã trích |
|---|---:|---|
| Checkpoint contract | 4 | `11/cell_04_checkpoint_backup_readiness.py` |
| 20-epoch training and backup | 5 | `11/cell_05_training_validation_backup.py` |
| Seven-model analysis | 6 | `11/cell_06_seven_model_statistics.py` |
| Selection freeze | 7 | `11/cell_07_freeze_selection.py` |
| One-time final test | 9 | `11/cell_09_final_test_once.py` |
| Final CI display | 10 | `11/cell_10_bootstrap_ci_report.py` |



## 7. Lưu ý tái lập và giới hạn

- **Đây không phải mã tái huấn luyện đã được kiểm thử sau refactor.** Muốn tái chạy phải xác minh phiên bản thư viện, split, SHA256 checkpoint nguồn, thứ tự thao tác và cơ chế backup.
- Những lần khôi phục phiên Kaggle có thể ảnh hưởng thứ tự RNG của DataLoader; checkpoint tại ranh giới epoch không cam kết bitwise identity.
- Không sử dụng kết quả từ test để điều chỉnh tham số sau khi đã khóa mô hình.
- Các mô hình khác kiến trúc (đặc biệt M6) phải được so sánh cả chi phí tính toán, không chỉ Dice.


## 8. Đánh giá cuối cùng được khóa (chỉ M6)

- **Model selection:** epoch 20 của M6 trên 55 ca validation, trước khi đọc 56 ca test.
- **Final 56-case internal test:** WT `0.871992437`, TC `0.808353757`, ET `0.723875037`, Mean Dice `0.801407077`.
- **Bootstrap 95% CI:** WT `[0.849027628, 0.892578090]`, TC `[0.757232951, 0.852441767]`, ET `[0.648603617, 0.790751592]`, Mean `[0.762349958, 0.836852589]`.
- Có 4 ca ET target rỗng; quy ước Dice của pipeline phải được trích dẫn khi báo cáo.
- **Checkpoint SHA256:** `02a2930122c3994fa72342921c719f1f001d540f4fb2e106fe293607ed527421`.
- Test lịch sử từng tồn tại trong repository; 'held-out' chỉ đúng với quy trình lựa chọn M1–M6 hiện tại, không đồng nghĩa test chưa từng được xem trong toàn dự án.

**Kaggle:** `tthanhh811/brats2020-m6-epoch20-seed42`, `tthanhh811/brats2020-m6-final-selection-seed42`, `tthanhh811/brats2020-m6-finaltest-56cases-seed42`.


In [ ]:
# Read-only: inspect fixed metadata from this review notebook.
# Safe to run locally; does not connect to Kaggle or load MRI.
experiment = 'M6'
validation_mean_dice = 0.775990758
validation_cases = 55
print(experiment, "| validation patients:", validation_cases, "| Mean Dice:", validation_mean_dice)


## 9. Quy trình đưa lên GitHub

1. Giữ `original_notebooks` làm bằng chứng lịch sử; không thay thế/ghi đè bản gốc.
2. Chỉ chuyển module mới vào `src/` sau khi đối chiếu hàm/class nguyên bản và chạy smoke test.
3. Đồng bộ kết quả tổng hợp và checksum; **không commit `.pt`, `.nii`, credential, hoặc dữ liệu bệnh nhân chưa được duyệt công khai**.
4. Tạo PR riêng để review thay đổi trước khi merge vào `main`.
